# Assignment 6: An Interactive Map of the Results

In the **sixth module** we met **Folium** and learned to build interactive web maps.

In this assignment you will bring together, on a single interactive map, the data you prepared at every earlier stage of the project.

The map should show:

- the boundary of the study area;
- land use;
- the everyday amenities and the public transport stops;
- the 5, 10 and 15-minute walking catchment areas around them (use the merged zones prepared in assignments 4 and 5).

Along the way you will:

- load the prepared spatial data;
- create a base map;
- add several thematic layers;
- add map controls;
- save the map as an `.html` file.

> We recommend working through the library's own documentation and examples as you go: [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html)

> The code cells below are placeholders for your own work: they are left unexecuted on purpose. Fill them in as you go, and run them in your own copy of the notebook.


## Suggested Steps

### Step 0. Importing Libraries

Import the libraries you will need.


In [ ]:
# your code


### Step 1. Loading the Data

Everything you need is in `project.gpkg`, built up over the previous assignments. Start by listing its layers with `gpd.list_layers()` to see what you have, then load:

| Layer | From |
| --- | --- |
| `area` | assignment 1 |
| `poi_<category>` | assignment 1 |
| `landuse` | assignment 2 |
| `stops`, `stops_catchment` | assignment 3 |
| `iso_<category>` | assignment 4, with the population figures added in assignment 5 |

Keep the categories in a dictionary again, the way you did in assignment 1 — the loops at the end of this notebook depend on it.

In [ ]:
# your code

# gpd.list_layers("project.gpkg")

# area = gpd.read_file("project.gpkg", layer="area")

### Step 2. Checking and Preparing the Data

Look at the structure of each layer. Drop the columns you do not need, keeping the ones used for display and for the tooltips.

**Check the CRS of every layer and bring them all to EPSG:4326.** Leaflet — and therefore Folium — works in degrees of latitude and longitude. `folium.GeoJson()` reprojects a `GeoDataFrame` for you, but a manual marker loop does not:

```python
folium.CircleMarker(location=[row.geometry.y, row.geometry.x])
```

reads the raw coordinates, so a layer left in UTM lands somewhere off the coast of Africa with no error and no warning. If you followed the storage rule from assignment 1 your layers are already in EPSG:4326; if you saved anything in UTM, reproject it now.

In [ ]:
# your code


### Step 3. Creating the Base Map

Work out the centre of the map from the boundary of the study area. Remember that the centroid should be calculated in a projected CRS and then converted back to degrees — Folium expects latitude and longitude.


In [ ]:
# your code


Create the map object.


In [ ]:
# your code


### Step 4. Adding the Study Area Boundary

Create a style for the boundary.


In [ ]:
# your code


Add the boundary to the map as a layer of its own.


In [ ]:
# your code


### Step 5. Adding the Land Use Layer

Look at which land use types the data contains.


In [ ]:
# your code


Create a dictionary of colours for the main land use types.


In [ ]:
# your code


Write the style function.


In [ ]:
# your code


Add tooltips if you need them.


In [ ]:
# your code


Add the layer to the map.


In [ ]:
# your code


#### Adding a Legend

A map with several land use colours is hard to read without a legend.

Folium has no ready-made legend for `GeoJson` layers, so it is added as a block of HTML on top of the map. The approach is shown in [Web Map with Folium](../module_6/map_1.ipynb), section 3.5 — including why the legend is generated from the same colour dictionary that drives the styling, and why it is attached to the map root rather than added as a layer.

Adapt it to your own categories.


In [ ]:
# your code


### Step 6. Adding the Features and Their Catchment Areas

Now add the features and the walking catchment areas that belong to them.

For each category, we recommend creating a separate layer group (`FeatureGroup`) holding:

- the features themselves;
- the 5-minute catchment area;
- the 10-minute catchment area;
- the 15-minute catchment area.

The catchment areas of a category can live in one layer or be split by interval — pick whichever is easier to manage and to read.


#### 6.1. The First Category

Below is an example of adding one category of features together with its catchment areas.


##### Creating the Layer Group


In [ ]:
# category_group = folium.FeatureGroup(
#     name="Schools and their catchment areas",
#     show=True
# )


`show=True` means the layer is visible as soon as the map loads. Think about whether every category should be on by default: when there are many layers, it is better to hide some of them with `show=False`.


##### Adding the Features

`CircleMarker` is a convenient choice for point features, but the documentation covers other options too:

- `Marker`;
- `Icon`;
- `BeautifyIcon`;
- Font Awesome icons.


In [ ]:
# for _, row in category_gdf.iterrows():

#     folium.CircleMarker(
#         location=[
#             row.geometry.y,
#             row.geometry.x
#         ],
#         radius=5,
#         color="blue",
#         fill=True,
#         fill_color="blue",
#         fill_opacity=0.9,
#         popup=row["name"],
#         tooltip="School"
#     ).add_to(category_group)


##### Adding the Catchment Areas

What you do next depends on how you saved the catchment areas:

- as a single dataset;
- or as several layers split by time interval.

The important part is adding them to the same group as the features:

```python
.add_to(category_group)
```

For the catchment areas we recommend:

- one colour per category of features;
- different saturation or opacity for the 5, 10 and 15-minute intervals;
- a more transparent fill for the larger zones, so they do not hide the rest of the data.

If the population figures are already in the attribute table of the zones, they can go into the tooltips or the popups.


Here is an example for the case where all the catchment areas sit in one dataset — say a `GeoDataFrame` with a `minutes` column holding the intervals. Styles and tooltips can then be assigned automatically through functions.


Start with a dictionary of colours and opacities for the intervals:


In [ ]:
# iso_styles = {
#     5: {
#         "fillColor": "#08519c",
#         "color": "#08519c",
#         "fillOpacity": 0.7
#     },
#     10: {
#         "fillColor": "#3182bd",
#         "color": "#3182bd",
#         "fillOpacity": 0.5
#     },
#     15: {
#         "fillColor": "#9ecae1",
#         "color": "#9ecae1",
#         "fillOpacity": 0.3
#     }
# }


The style function then picks the style from the value of the `minutes` field.


In [ ]:
# def style_function(feature):

#     minutes_value = feature["properties"]["minutes"]

#     style = iso_styles[minutes_value]

#     return {
#         "fillColor": style["fillColor"],
#         "color": style["color"],
#         "weight": 1,
#         "fillOpacity": style["fillOpacity"]
#     }


If the data carries population figures, they can be shown in the tooltips:


In [ ]:
# tooltip = folium.GeoJsonTooltip(
#     fields=[
#         "minutes",
#         "population_inside",
#         "share_inside"
#     ],
#     aliases=[
#         "Walking time, min:",
#         "Population inside:",
#         "Share of population:"
#     ],
#     localize=True
# )


Adding the layer to the map:


In [ ]:
# folium.GeoJson(
#     isochrones_gdf,
#     name="School catchment areas",
#     style_function=style_function,
#     tooltip=tooltip
# ).add_to(category_group)


##### Adding the Group to the Map


In [ ]:
# category_group.add_to(m)


#### 6.2. The Remaining Categories

Once the first category looks right, repeat the same steps for the others. For each category we recommend creating its own `FeatureGroup`.


In [ ]:
# your code


#### 6.3. Automating the Process (optional)

After a couple of categories you will notice the code repeating itself.

Think about how to automate it. For example:

- keep the styles of the categories in a dictionary;
- move the adding of features into a function;
- move the adding of catchment areas into a function;
- loop over all the categories.


In [ ]:
# your code


### Step 7. Adding Map Controls


A layer switcher is essential — add it.


In [ ]:
# your code


You can add anything else you find useful:

- a mini map;
- a fullscreen button;
- a distance measuring tool;
- a legend;
- a feature search.


### Step 8. Viewing, Saving and Publishing the Map

Display the finished map.


In [ ]:
# your code


Save the map to an HTML file.


In [ ]:
# your code


Open the file in a browser and check that:

- the map loads;
- the layers can be switched on and off;
- the tooltips show what they should;
- the map reads well visually.


After that you can publish the map on GitHub Pages or anywhere else — the steps are in [Publishing on GitHub Pages](../module_6/map_2.ipynb).


### Step 9. Conclusions: What the Project Found

The map is the artefact; this step is the answer.

Pull the numbers from [assignment 5](../module_5/rasters_task.ipynb) into a short summary — the share of the population within a 15-minute walk of each category of amenity — and write three to five sentences covering:

1. **Which everyday needs are met** in your area within 15 minutes on foot, and which are not.
2. **Where the gaps are**: which parts of the area stay outside the catchment areas, and whether people actually live there — check against the `landuse` layer and the buildings from assignment 3.
3. **Which category is the weakest link.** If one category leaves a third of residents outside, that is the finding worth acting on.
4. **What you would do about it**: where a single new amenity would close the largest gap.
5. **How much to trust the result** — one sentence on the limitations you listed in assignment 5 (OSM completeness, modelled population, approximate catchment areas).

This is what a reader of your map should be able to take away without opening a single notebook.

> _your conclusions here_

## What You Should Have

By the end of this assignment you should have an interactive web map showing the data prepared at every stage of the project:

- the boundary of the study area;
- land use;
- the everyday amenities;
- the public transport stops;
- the walking catchment areas;
- map controls;
- the map saved as an HTML file;
- a short set of conclusions: which everyday amenities are within a 15-minute walk for what share of residents, and where the gaps are.